# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below, we list the `@id`s for found record sets and their sample fields. The `@id`s are critical for referencing the entities throughout the rest of this notebook as per Croissant best practices.

In [ ]:
# List all record sets with their @id

record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets explicitly listed in metadata; attempting to discover record sets from data files...")
    # Sometimes datasets with no explicit recordSet objects define just 1 data file as a single set
    # To explore, let's print all available records
    all_records = list(dataset.records())
    print(f"Found {len(all_records)} records. Example:")
    if len(all_records) > 0:
        print(all_records[0])
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}, fields: {[field.id for field in rs.fields]}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

We'll first attempt to discover a record set's `@id`, but if the dataset exposes only a single data file, we'll treat all records as belonging to a pseudo record set with the `@id` value of `'main'` for further referencing.

In [ ]:
# Extract records into DataFrame

record_sets = list(dataset.record_sets)
dataframes = {}
record_set_ids = []

if not record_sets:
    # Fallback: no recordSet objects, treat all records as belonging to an implicit set @id 'main'
    all_records = list(dataset.records())
    dataframes['main'] = pd.DataFrame(all_records)
    record_set_ids = ['main']
else:
    # Iterate through explicit record sets
    for rs in record_sets:
        record_set_id = rs.id
        records = list(dataset.records(record_set=record_set_id))
        dataframes[record_set_id] = pd.DataFrame(records)
        record_set_ids.append(record_set_id)

print(f"Available record set IDs: {record_set_ids}")

# Inspect columns available in the first record set
example_record_set_id = record_set_ids[0] if record_set_ids else None
if example_record_set_id and not dataframes[example_record_set_id].empty:
    print(f"Sample columns for record set '@id={example_record_set_id}':")
    print(dataframes[example_record_set_id].columns.tolist())
    display(dataframes[example_record_set_id].head())
else:
    print("No data available to display.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

First, we'll choose a numeric field (by `@id` from columns found above) and a grouping field if available.

In [ ]:
# EDA: Filtering, normalization, grouping

# You may need to adjust these based on what fields/columns are present in your dataset
# We'll make a best guess for field @id's typical of mass spectrometry datasets

record_set_id = example_record_set_id
df = dataframes[record_set_id]

# Let's try to find a numeric field (e.g., 'MW' for molecular weight, 'Abundance', 'Peptide_Counts', etc.)
possible_numeric_fields = [col for col in df.columns if any(x in col.lower() for x in ['mw', 'abundance', 'count', 'intensity', 'amount', 'number', 'coverage'])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]  # Pick the first as example
else:
    # Default to first numeric field type
    numeric_field = df.select_dtypes(include=['number']).columns[0] if not df.select_dtypes(include=['number']).empty else df.columns[0]

print(f"Selected numeric field for analysis: {numeric_field}")

try:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
except Exception:
    pass

# Filter out nulls for the selected field
df_filtered = df[df[numeric_field].notnull()]

# Choose a threshold for filtering (for demonstration, use 90th percentile as threshold, or 10 as default)
if len(df_filtered) > 0:
    threshold = df_filtered[numeric_field].quantile(0.9) if df_filtered[numeric_field].dtype in ['float64', 'int64'] else 10
else:
    threshold = 10

filtered_df = df_filtered[df_filtered[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()

print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Choose a field to group by if available (e.g., 'Sample', 'protein_class', 'description')
possible_group_fields = [col for col in df.columns if any(x in col.lower() for x in ['sample', 'class', 'description', 'type', 'group'])]
if possible_group_fields:
    group_field = possible_group_fields[0]
    print(f"Grouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field} (showing average {numeric_field}):")
    display(grouped_df.head())
else:
    group_field = None
    print("No suitable group field found to group data.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll build a histogram for the selected numeric field and, where possible, a bar plot of the grouped field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df_filtered[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If we have a group_field, plot the average per group (top 10 groups by mean)
if group_field is not None and 'grouped_df' in locals():
    top_groups = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=group_field, y=numeric_field, data=top_groups)
    plt.title(f"Top 10 {group_field}s by average {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.show()

## 6. Conclusion
In this notebook, we have successfully loaded the FAIR^2 mass spectrometry dataset, explored available fields using the entity `@id`s, and performed basic data filtering, normalization, grouping, and visualization.

Key observations:
- The dataset centers on protein abundance and modification analysis from human mast cell extracellular vesicles.
- Numeric analysis (such as MW or abundance) allows for filtering and statistical normalization.
- Top grouping fields may reveal the most abundant or prominent protein classes, sequences, or sample groups.

For further exploration, refer to the Croissant metadata and documentation for all supported fields and the structure of record sets.

---
**References:**
- Dataset: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
- Croissant documentation: https://mlcommons.org/croissant